# OpenAI API
## gpt-5-nano und gpt-5-mini sind günstiger und reiches völlig aus
## Aufbau von Vektordatenbank

# Version 1: verbesserte Version, vergleichen mit der ersten Version

In [2]:
import os
import re
from typing import List, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv  
import shutil

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

# =====================================================================
# CONFIGURATION & API SETUP
# =====================================================================
load_dotenv()  
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Fehler: Kein OPENAI_API_KEY in der .env-Datei gefunden!")
    
PERSIST_DIRECTORY = "chroma_mietrecht_parapraph_rag"
DOCUMENTS_DIR = "dokumente_mietrecht_paragraph"

os.makedirs(DOCUMENTS_DIR, exist_ok=True)

# 1. TEXT EMBEDDINGS
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. DUAL-STACK LLM SETUP
llm_nano = ChatOpenAI(model="gpt-5.4-nano", temperature=0.1)

# Stack 2: Für hohe logische Tiefe und exzellenten Citation Recall (Quellen)
llm_mini = ChatOpenAI(model="gpt-5.4-mini", temperature=0.0)

# Stack 1: Für die schnelle Filter-Extraktion
#llm_nano = ChatOpenAI(model="gpt-5-nano", temperature=0.1)

# Stack 2: Für die finale juristische Antwort
#llm_mini = ChatOpenAI(model="gpt-5-mini", temperature=0.0)
# =====================================================================
# SYSTEMATISCHES EINLESEN
# =====================================================================
if 'vector_store' not in globals():
    print("🤖 Initialisiere saubere Chroma-Datenbank mit OpenAI Embeddings...")
    vector_store = Chroma(
        persist_directory=PERSIST_DIRECTORY,
        embedding_function=embeddings
    )
else:
    print("🔄 Nutze bestehende Verbindung.")

# Bereits verarbeitete Dateien ermitteln
existierende_daten = vector_store.get()
bereits_eingelesen = set()
if existierende_daten and "metadatas" in existierende_daten:
    for meta in existierende_daten["metadatas"]:
        if meta and "source" in meta:
            bereits_eingelesen.add(os.path.basename(meta["source"]))

alle_dateien = [f for f in os.listdir(DOCUMENTS_DIR) if f.endswith(".txt")]
neue_dateien = [f for f in alle_dateien if f not in bereits_eingelesen]

if neue_dateien:
    print(f"✨ Neue Dokumente gefunden: {neue_dateien}")
    for datei_name in neue_dateien:
        datei_pfad = os.path.join(DOCUMENTS_DIR, datei_name)
        print(f"📄 Indiziere: {datei_name}...")
        
        try:
            with open(datei_pfad, "r", encoding="utf-8") as f:
                volltext = f.read()
            
            # Präziser Split nach deinen Paragraphen-Markern
            # sections = re.split(r"(§\s*\d+[a-zA-Z]*\s*[^\n]*)", text) allgemein nach §

            rohe_paragraphen = volltext.split("=== PARAGRAPH_START:")
            #print("rohe_paragraphen:", rohe_paragraphen)
            chunks = []
            
            for raw_chunk in rohe_paragraphen:
                text_inhalt = raw_chunk.strip()
                if not text_inhalt:
                    continue
                if "=== PARAGRAPH_END ===" in text_inhalt:
                    text_inhalt = text_inhalt.split("=== PARAGRAPH_END ===")[0].strip()
                
                # Robustere Extraktion der Paragraphen-Nummer (z.B. "573c")
                p_match = re.search(r"§\s*(\d+[a-z]*)", text_inhalt)
                paragraph_meta = p_match.group(1) if p_match else "unbekannt"
                
                metadata = {
                    "source": datei_name,
                    "paragraph": paragraph_meta  
                }
                
                chunks.append(Document(page_content=text_inhalt, metadata=metadata))
                     
            if chunks:
                vector_store.add_documents(documents=chunks)
                print(f"✅ {datei_name} erfolgreich mit {len(chunks)} Chunks eingelesen.")
        except Exception as e:
            print(f"❌ Fehler bei {datei_name}: {e}")
else:
    print("ℹ️ Vektordatenbank ist auf dem neuesten Stand.")

# =====================================================================
# STACK 1: QUERY TRANSFORMATION STRUCTURE
# =====================================================================
class SearchQueryExtraction(BaseModel):
    search_phrase: str = Field(description="Die optimierte, juristische Suchphrase.")
    paragraph_filters: List[str] = Field(
        default_factory=list, 
        description="Eine Liste der reinen Paragraphennummern (z.B. ['573', '573c']). Wenn keine spezifischen Paragraphen genannt oder impliziert werden, gib eine leere Liste [] aus."
    )

query_transform_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Du bist ein präziser KI-Assistent für deutsches Mietrecht.\n"
        "Analysiere die Nutzeranfrage und extrahiere:\n"
        "1. Eine prägnante, juristische Suchphrase für die Vektorsuche.\n"
        "2. Alle relevanten Paragraphennummern (NUR Nummern/Buchstaben, z.B. ['573', '573c']) als Liste.\n"
        "Nutze leere Listen [], wenn kein konkreter Paragraph zuzuordnen ist."
    )),
    ("human", "Nutzeranfrage: {original_query}")
])

# Kette für die Extraktion über das Nano-Modell
query_transform_chain = query_transform_prompt | llm_nano.with_structured_output(SearchQueryExtraction)

# =====================================================================
# DYNAMIC RETRIEVAL & QA SETUP
# =====================================================================
system_prompt = (
    "Du bist ein präziser Rechtsassistent für deutsches Mietrecht.\n"
    "Dir werden mehrere Paragraphen als Kontext bereitgestellt. Sie enthalten jeweils "
    "[ORIGINALTEXT], [ERKLÄRUNG] und [TYPISCHE_NUTZERFRAGEN].\n\n"
    "Prüfe alle übergebenen Abschnitte sorgfältig. Beantworte die Frage des Nutzers "
    "wahrheitsgemäß und nenne zwingend den exakten Paragraphen aus dem [ORIGINALTEXT] als Quelle.\n"
    "Wenn die Antwort im bereitgestellten Kontext nicht zu finden ist, sage klipp und klar:\n"
    "'Ich weiß es nicht, da der passende Gesetzestext im Kontext fehlt.'\n\n"
    "Kontext:\n{context}"
    "WICHTIG: Antworte in reinem, flüssigem Text. Nutze KEINERLEI Markdown-Formatierungen "
    "wie Sternchen (**), Rauten (#) oder Listen-Unterstriche."
)




qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

def dynamic_retrieval(extracted: SearchQueryExtraction):
    """Nimmt die Struktur von Nano und konfiguriert den Filter für Chroma live."""
    print("\n--- RAG RETRIEVAL START ---")
    print(f"🔍 Nano-Suchphrase: '{extracted.search_phrase}'")
    print(f"📋 Nano-Filterliste: {extracted.paragraph_filters}")
    
    anzahl_paragraphen = len(extracted.paragraph_filters)
    dynamisches_k = max(2, anzahl_paragraphen * 2)
    
    search_kwargs = {
        "k": dynamisches_k,
        "score_threshold": 0.4
    }
    
    if extracted.paragraph_filters:
        if len(extracted.paragraph_filters) == 1:
            print(f"🎯 Filter aktiv: Suche nur in § {extracted.paragraph_filters[0]}")
            search_kwargs["filter"] = {"paragraph": extracted.paragraph_filters[0]}
        else:
            print(f"🎯 Multi-Filter aktiv: Suche in {extracted.paragraph_filters}")
            search_kwargs["filter"] = {"paragraph": {"$in": extracted.paragraph_filters}}
    else:
        print("🌐 Kein spezifischer Filter aktiv. Nutze Standardsuche.")
        
    print("---------------------------\n")
    
    dynamic_retriever = vector_store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs=search_kwargs
    )
    
    return dynamic_retriever.invoke(extracted.search_phrase)

def generate_final_response(input_dict):
    """Validiert den Schwellenwert und generiert die Antwort über GPT-5.4 Mini."""
    docs = input_dict["context"]
    question = input_dict["question"]
    
    # Abfangen falls score_threshold nichts durchlässt
    if not docs:
        return ("Ich weiß es nicht, da der passende Gesetzestext im Kontext fehlt. "
                "(Grund: Es wurden keine Dokumente gefunden, die eine ausreichende "
                "Ähnlichkeit mit Ihrer Frage aufweisen.)")
    
    formatted_context = "\n\n".join(
        f"Quelle: {doc.metadata.get('source', 'Unbekannt')} (§ {doc.metadata.get('paragraph', 'unbekannt')})\nContent:\n{doc.page_content}" 
        for doc in docs
    )
    
    final_prompt = qa_prompt.format(context=formatted_context, question=question)
    
    # Korrektur: Nutze das starke Mini-Modell für die juristische Begründung
    return llm_mini.invoke(final_prompt).content

# =====================================================================
# DIE INTELLIGENTE LCEL-KETTE (KORRIGIERT)
# =====================================================================
rag_chain = (
    # 1. Schritt: Wir isolieren die 'original_query' und führen die Transformation parallel aus
    RunnablePassthrough.assign(
        extracted_meta=lambda x: query_transform_chain.invoke({"original_query": x["original_query"]})
    )
    # 2. Schritt: Wir mappen den Context (via dynamic_retrieval) und behalten die Ur-Frage bei
    | {
        "context": lambda x: dynamic_retrieval(x["extracted_meta"]),
        "question": lambda x: x["original_query"]
    }
    # 3. Schritt: Finale Weiche und Antwort-Generierung via llm_mini
    | RunnableLambda(generate_final_response)
)

# =====================================================================
# TESTLAUF
# =====================================================================
if __name__ == "__main__":
    such_query = "Ich wohne seit 6 Jahren in meiner Wohnung. Welche Kündigungsfrist gilt für meinen Vermieter?"
    finale_antwort = rag_chain.invoke({"original_query": such_query})
    print("\n=== ANTWORT ===")
    print(finale_antwort)

🤖 Initialisiere saubere Chroma-Datenbank mit OpenAI Embeddings...
✨ Neue Dokumente gefunden: ['mietrecht_kuendigung_ganz_para.txt', 'mietrecht_mietzahlung_ganz_para.txt']
📄 Indiziere: mietrecht_kuendigung_ganz_para.txt...
✅ mietrecht_kuendigung_ganz_para.txt erfolgreich mit 4 Chunks eingelesen.
📄 Indiziere: mietrecht_mietzahlung_ganz_para.txt...
✅ mietrecht_mietzahlung_ganz_para.txt erfolgreich mit 8 Chunks eingelesen.

--- RAG RETRIEVAL START ---
🔍 Nano-Suchphrase: 'Kündigungsfrist Vermieter bei langjähriger Mietdauer (6 Jahre) welche Frist gilt'
📋 Nano-Filterliste: []
🌐 Kein spezifischer Filter aktiv. Nutze Standardsuche.
---------------------------


=== ANTWORT ===
Für Ihren Vermieter gilt nach § 573c Absatz 1 BGB eine Kündigungsfrist von sechs Monaten, wenn seit der Überlassung des Wohnraums fünf Jahre vergangen sind; nach acht Jahren verlängert sie sich auf neun Monate. Da Sie seit 6 Jahren dort wohnen, beträgt die Kündigungsfrist für den Vermieter sechs Monate. Quelle: § 573c Ab

# andere User-Frage

In [7]:

if __name__ == "__main__":
    such_query = "Ich wohne seit 3 Jahren in meiner Wohnung. Welche Kündigungsfrist gilt für meinen Vermieter?"
    finale_antwort = rag_chain.invoke({"original_query": such_query})
    print("\n=== ANTWORT ===")
    print(finale_antwort)


--- RAG RETRIEVAL START ---
🔍 Nano-Suchphrase: 'Kündigungsfrist Vermieter bei Mietverhältnis seit 3 Jahren; allgemeine Kündigungsfristen nach BGB; Kündigung wegen Eigenbedarfs bzw. ordentliche Kündigung'
📋 Nano-Filterliste: ['573', '573c', '573d']
🎯 Multi-Filter aktiv: Suche in ['573', '573c', '573d']
---------------------------


=== ANTWORT ===
Für deinen Vermieter gilt nach § 573c Absatz 1 BGB grundsätzlich eine Kündigungsfrist von drei Monaten. Der exakte Gesetzestext lautet: „Die Kündigungsfrist für den Vermieter verlängert sich nach fünf und acht Jahren seit der Überlassung des Wohnraums um jeweils drei Monate.“ Da du erst seit 3 Jahren in der Wohnung wohnst, ist diese Verlängerung noch nicht einschlägig. Quelle: § 573c Absatz 1 BGB.


# Documentation und Alte Textversion

In [ ]:


#Das neue Zusammenspiel in der LCEL-Kette
# So verbindest du die neuen Bausteine (Metadaten-Filter und Abfang-Schutz) in deiner bestehenden Kette:
# =====================================================================
# STACK 2: FINALE JURISTISCHE ANTWORT (GPT-5.4 Mini)
# =====================================================================
# Das logisch tiefere Modell schreibt die Antwort und achtet penibel auf die Quellen.
system_prompt = (
    "Du bist ein präziser Rechtsassistent für deutsches Mietrecht.\n"
    "Dir werden mehrere Paragraphen als Kontext bereitgestellt. Sie enthalten jeweils "
    "[ORIGINALTEXT], [ERKLÄRUNG] und [TYPISCHE_NUTZERFRAGEN].\n\n"
    "Prüfe alle übergebenen Abschnitte sorgfältig. Beantworte die Frage des Nutzers "
    "wahrheitsgemäß und nenne zwingend den exakten Paragraphen aus dem [ORIGINALTEXT] als Quelle.\n"
    "Wenn die Antwort im bereitgestellten Kontext nicht zu finden ist, sage klipp und klar: "
    "'Ich weiß es nicht, da der passende Gesetzestext im Kontext fehlt.'\n\n"
    "Kontext:\n{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

def format_docs(docs):
    # Hilfsfunktion, um die gefundenen Dokumente sauber zu formatieren
    return "\n\n".join(f"Quelle: {doc.metadata.get('source', 'Unbekannt')}\nContent:\n{doc.page_content}" for doc in docs)

# Aufbau der modernen LCEL RAG-Kette mit integrierter Dual-Stack-Logik
rag_chain = (
    {
        "context": query_transform_chain | retriever | format_docs,
        "question": lambda x: x["original_query"]
    }
    | qa_prompt
    | llm_nano # llm_mini
    | StrOutputParser()
)
# print( "rag_chain:", rag_chain)

# =====================================================================
# EXECUTION (ABFRAGE)
# =====================================================================
such_query = "Ich wohne seit 6 Jahren in meiner Wohnung. Welche Kündigungsfrist gilt für meinen Vermieter?"
print(f"\nOriginale Frage: {such_query}")

# Schritt 1 simulieren: Was macht Nano aus der Frage?
optimierte_suche = query_transform_chain.invoke({"original_query": such_query})
print(f"🤖 [Stack 1 - Nano] Optimierte Suchanfrage: '{optimierte_suche}'\n")

# 2. Chunks manuell aus der Datenbank abrufen
gefundene_chunks = retriever.invoke(optimierte_suche)

# 3. Chunks übersichtlich in der Konsole ausgeben
print("\n=== GEFUNDENE CHUNKS AUS DER DATENBANK ===")
for i, doc in enumerate(gefundene_chunks, 1):
    print(f"\n📄 [Chunk {i}] Quelle: {doc.metadata.get('source')}")
    print("-" * 40)
    print(doc.page_content)
    print("-" * 40)




# EXECUTION (ABFRAGE)

In [ ]:


print("Berechne finale Antwort mit Mini...")
# Schritt 2: Gesamtes RAG-System durchlaufen
finale_antwort = rag_chain.invoke({"original_query": such_query})

print("\n=== ANTWORT VON GPT-5.4 MINI ===")
print(finale_antwort)



# Python Übung 1

In [14]:
# 1. Die Schablone (Definition): Text mit Platzhaltern in geschweiften Klammern
prompt_bauplan = [
    ("system", "Du bist ein KI-Assistent für das Thema: {thema}."),
    ("human", "Hier ist meine Frage: {original_query}")
]

print(prompt_bauplan)
# 2. Die Daten (Das Dictionary): Hier stecken die echten Werte drin
benutzer_daten = {
    "thema": "Mietrecht",
    "original_query": "Darf mein Vermieter die Haustiere verbieten?"
}

# 3. Das Befüllen (Die Kette): Wir bauen die finale Liste zusammen
fertige_nachrichten = []

for rolle, text in prompt_bauplan:
    print("1 rolle:", rolle)
    print("2 text:", text)
    print("3 type(text):", type(text))
    # .format(**benutzer_daten) sucht im Dictionary nach den passenden Schlüsseln
    befuellter_text = text.format(**benutzer_daten)
    fertige_nachrichten.append((rolle, befuellter_text))

# 4. Das Ergebnis anzeigenprint()
print()
print(fertige_nachrichten)


[('system', 'Du bist ein KI-Assistent für das Thema: {thema}.'), ('human', 'Hier ist meine Frage: {original_query}')]
1 rolle: system
2 text: Du bist ein KI-Assistent für das Thema: {thema}.
3 type(text): <class 'str'>
1 rolle: human
2 text: Hier ist meine Frage: {original_query}
3 type(text): <class 'str'>

[('system', 'Du bist ein KI-Assistent für das Thema: Mietrecht.'), ('human', 'Hier ist meine Frage: Darf mein Vermieter die Haustiere verbieten?')]


# Übung 2

In [8]:
import re

text_inhalt = "Gemäß § 123b des Strafgesetzbuches ist dies geregelt."

# Suche ausführen
p_match = re.search(r"§\s*(\d+[a-z]*)", text_inhalt)
print(p_match.group(0))  # Ausgabe: "§ 123b"
print(p_match.group(1))  # Ausgabe: "§ 123b"
print(p_match.group())  # Ausgabe: "§ 123b"
p_match


§ 123b
123b
§ 123b


<re.Match object; span=(6, 12), match='§ 123b'>

# Übung 3

In [14]:
import re
from langchain_core.documents import Document

def enrich_legal_text(text: str) -> str:
    """
    Macht Gesetzestexte RAG-freundlich ohne Inhalt zu verlieren.
    """

    sections = re.split(r"(§\s*\d+[a-zA-Z]*\s*[^\n]*)", text)
    print("sections:", sections)
    enriched_chunks = []

    current_section = ""

    for part in sections:
        if re.match(r"§\s*\d+", part):
            current_section = part.strip()
        else:
            content = part.strip()
            if not content:
                continue

            # einfache heuristische Frage-Generierung
            questions = []

            if "Kündigungsfrist" in content:
                questions.append("Wie lange ist die Kündigungsfrist?")
                questions.append("Welche Fristen gelten für Vermieter?")

            if "berechtigtes Interesse" in content:
                questions.append("Wann darf der Vermieter kündigen?")

            enriched_text = f"""
{current_section}

[USER-FRAGEN]
- {chr(10).join(questions)}

[RECHTSTEXT]
{content}
"""

            enriched_chunks.append(enriched_text)

    return enriched_chunks

text_inhalt = "§ 123b des Strafgesetzbuches ist dies geregelt."
r_chunks= enrich_legal_text(text_inhalt)

print(r_chunks)



sections: ['', '§ 123b des Strafgesetzbuches ist dies geregelt.', '']
[]
